# paper_ready_sections

## 0. Setup
本notebook用于论文主文的可复现图表与表格生成。

**Reproducibility**
- Project root: `~/Pr/power-graph-risk-learning`
- Primary dataset: `data/processed/downstream/downstream_v2_informative.parquet`
- Main results: `analysis/training/v2_final_summary.json`, `paper_main_results_final.csv`
- Evaluation protocol: strict Leave-One-Network-Out (LONO)


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, precision_recall_curve, auc

# robust project-root detection (works whether cwd is repo root or notebook dir)
_cwd = Path.cwd().resolve()
ROOT = _cwd
for cand in [_cwd, *_cwd.parents]:
    if (cand / 'data/processed/downstream/downstream_v2_informative.parquet').exists():
        ROOT = cand
        break

DATA_PATH = ROOT / 'data/processed/downstream/downstream_v2_informative.parquet'
OUT_DIR = ROOT / 'analysis/training'
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_PATH)
print('ROOT=', ROOT)
print('rows=', len(df), 'cols=', len(df.columns), 'networks=', sorted(df['network'].unique()))

---
## 1. Problem & Setting

- Cascading failure risk prediction under **cross-topology domain shift**.
- Random split overestimates generalization; use strict **LONO**.

### LONO定义（必须）
设网络集合为 $\mathcal{N}=\{n_1,\dots,n_K\}$，第 $k$ 折训练/测试为：
$$
\mathcal{D}_{train}^{(k)} = \bigcup_{n \in \mathcal{N} \setminus \{n_k\}} \mathcal{D}_n,
\quad
\mathcal{D}_{test}^{(k)} = \mathcal{D}_{n_k}
$$
最终指标取 $K$ 折均值。

### 建议图
- 不同 network 的分布可视化（PCA / t-SNE）。


In [ ]:
# Distribution plot by network (PCA)
feature_cols = [c for c in df.columns if c.startswith('ef_') or c.startswith('efnc_') or c.endswith('_tshift')]
meta_cols = [c for c in ['n_samples','pos_rate','yreg_mean','yreg_std'] if c in df.columns]
use_cols = feature_cols + meta_cols

# subsample for plotting speed (preserve network column across pandas versions)
parts = []
for n, g in df.groupby('network'):
    gg = g.sample(min(2000, len(g)), random_state=42).copy()
    gg['network'] = n
    parts.append(gg)
plot_df = pd.concat(parts, ignore_index=True)
X = plot_df[use_cols].astype(float).values
X = StandardScaler().fit_transform(X)

pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(X)

plt.figure(figsize=(7,5))
for n, g in plot_df.groupby('network'):
    idx = g.index
    plt.scatter(Z[idx,0], Z[idx,1], s=6, alpha=0.5, label=n)
plt.title('Network distribution (PCA)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(markerscale=2)
plt.tight_layout()
plt.show()

---
## 2. Dataset

### v2设计（5点）
1. 多级风险标签（risk level + `y_cls_v2`）
2. 难例增强（hard example augmentation）
3. 时间扰动代理特征（`*_tshift`）
4. 网络级元特征（`n_samples`, `pos_rate`, `yreg_mean`, `yreg_std`）
5. 按网络稳健归一化（median / IQR）

### 必要表
- dataset summary（network / samples / pos rate）


In [ ]:
# Dataset summary table
summary = df.groupby('network').agg(
    samples=('network','size'),
    pos_rate=('y_cls_v2','mean'),
    yreg_mean=('y_reg','mean'),
    yreg_std=('y_reg','std')
).reset_index().sort_values('network')
summary['pos_rate'] = summary['pos_rate'].round(4)
summary.to_csv(OUT_DIR / 'paper_dataset_summary.csv', index=False)
summary

In [ ]:
# Risk distribution
plt.figure(figsize=(7,4))
vals = np.clip(df['y_reg'].astype(float).values, 0, np.quantile(df['y_reg'].astype(float).values, 0.99))
plt.hist(vals, bins=80, alpha=0.8)
plt.title('Risk distribution (y_reg clipped at 99th percentile)')
plt.xlabel('y_reg')
plt.ylabel('count')
plt.tight_layout()
plt.show()

---
## 3. Features & Normalization

- Temporal-like features: `*_tshift`
- Meta features: network-level statistics
- Robust normalization by network

### 公式
对每个特征 $x$ 在网络 $n$ 内进行：
$$
\tilde{x} = \frac{x - \mathrm{median}_n(x)}{\mathrm{IQR}_n(x)+\epsilon}
$$


In [ ]:
# Normalization before/after example (first feature)
f = use_cols[0]
raw = df[[f,'network']].copy()

def robust_norm(g):
    med = g[f].median()
    iqr = g[f].quantile(0.75)-g[f].quantile(0.25)
    iqr = iqr if iqr > 1e-9 else 1.0
    g['norm'] = (g[f]-med)/iqr
    return g

parts = []
for n, g in raw.groupby('network'):
    gg = robust_norm(g.copy())
    gg['network'] = n
    parts.append(gg)
norm = pd.concat(parts, ignore_index=True)

fig, ax = plt.subplots(1,2, figsize=(10,4))
for n, g in raw.groupby('network'):
    ax[0].hist(g[f].values, bins=50, alpha=0.3, label=n)
ax[0].set_title(f'Before normalization: {f}')
for n, g in norm.groupby('network'):
    ax[1].hist(g['norm'].values, bins=50, alpha=0.3, label=n)
ax[1].set_title(f'After robust normalization: {f}')
for a in ax: a.legend(fontsize=7)
plt.tight_layout()
plt.show()

---
## 4. Model

- Classifier: RandomForest (weighted positives)
- Regressor: ExtraTrees on $\log(1+y_{reg})$

### 建议公式
$$
\hat{y}=\frac{1}{T}\sum_{t=1}^{T} f_t(x)
$$
其中 $f_t$ 为第 $t$ 棵树。

### 模型对比
- Baseline / v2 / PU+multitask / GraphMAE+DANN/CORAL


In [ ]:
# Load existing result summaries for model comparison
files = [
    'v2_final_summary.json',
    'pu_multitask_v1_summary.json',
    'v2_graphmae_dann_coral_summary.json',
    'v2_graphmae_dann_coral_staged_summary.json',
]
rows = []
for fn in files:
    p = OUT_DIR / fn
    if p.exists():
        d = json.loads(p.read_text())
        c = d.get('classification_mean', d.get('best_balanced_classification', {}))
        r = d.get('regression_mean', {})
        rows.append({'model': fn.replace('_summary.json',''),
                     'auc': c.get('auc'), 'ap': c.get('ap'), 'f1': c.get('f1'), 'recall': c.get('recall'),
                     'mae': r.get('mae'), 'rmse': r.get('rmse')})
cmp_df = pd.DataFrame(rows)
cmp_df

---
## 5. Evaluation (LONO)

- Why not random split: random split leaks topology-specific cues, inflating performance.
- LONO better approximates deployment on unseen networks.

### 必要表
- LONO split table


In [ ]:
# LONO split table
nets = sorted(df['network'].unique())
split_rows = []
for te in nets:
    split_rows.append({'fold_test_network': te, 'train_networks': ', '.join([n for n in nets if n != te])})
split_df = pd.DataFrame(split_rows)
split_df.to_csv(OUT_DIR / 'paper_lono_splits.csv', index=False)
split_df

---
## 6. Results ⭐核心

此处放论文主结果段落（可直接引用 `paper_ready_sections_2026-04-04.md` 内容）。

### 必要表
- Main results: AUC / AP / F1 / Recall / MAE

### 必要图
- ROC
- PR


In [ ]:
# Main results table
main_csv = OUT_DIR / 'paper_main_results_final.csv'
if main_csv.exists():
    main_df = pd.read_csv(main_csv)
else:
    main_df = pd.DataFrame()
main_df

In [ ]:
# ROC / PR curves from sample-level LONO scores
score_path = OUT_DIR / 'v2_lono_scores.parquet'
if not score_path.exists():
    raise FileNotFoundError(f'Missing score file: {score_path}. Run scripts/export_v2_lono_scores.py first.')

sdf = pd.read_parquet(score_path)
y_true = sdf['y_true'].astype(int).values
y_score = sdf['y_score'].astype(float).values

fpr, tpr, _ = roc_curve(y_true, y_score)
prec, rec, _ = precision_recall_curve(y_true, y_score)
auc_roc = auc(fpr, tpr)
auc_pr = auc(rec, prec)

fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].plot(fpr, tpr, label=f'ROC AUC={auc_roc:.4f}')
ax[0].plot([0,1],[0,1],'--',alpha=0.5)
ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].set_title('ROC (LONO pooled)')
ax[0].legend()

ax[1].plot(rec, prec, label=f'PR AUC={auc_pr:.4f}')
base = y_true.mean()
ax[1].hlines(base, 0, 1, linestyles='--', alpha=0.5, label=f'Base rate={base:.4f}')
ax[1].set_xlabel('Recall'); ax[1].set_ylabel('Precision'); ax[1].set_title('PR (LONO pooled)')
ax[1].legend()
plt.tight_layout()
plt.show()

pd.DataFrame({'fpr': fpr, 'tpr': tpr}).to_csv(OUT_DIR / 'paper_roc_curve.csv', index=False)
pd.DataFrame({'recall': rec, 'precision': prec}).to_csv(OUT_DIR / 'paper_pr_curve.csv', index=False)
print({'roc_auc': float(auc_roc), 'pr_auc': float(auc_pr), 'base_rate': float(base)})

---
## 7. Ablation

- v2 vs baseline / threshold / adaptation variants

### 必要表
- ablation 对比

### 建议图
- bar chart


In [ ]:
# Ablation table + bar chart
if 'cmp_df' in globals() and not cmp_df.empty:
    abl = cmp_df.sort_values('auc', ascending=False)
    display(abl)

    plt.figure(figsize=(8,4))
    x = np.arange(len(abl))
    plt.bar(x-0.2, abl['auc'], width=0.2, label='AUC')
    plt.bar(x, abl['ap'], width=0.2, label='AP')
    plt.bar(x+0.2, abl['f1'], width=0.2, label='F1')
    plt.xticks(x, abl['model'], rotation=30, ha='right')
    plt.ylim(0, 1)
    plt.title('Ablation comparison (classification metrics)')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No ablation summary files found.')